# 02 — Data Preprocessing

**Purpose:** Clean and prepare the TMDB dataset for all three recommender algorithms.

**Input:** Raw CSV files in `data/raw/`
**Output:** Cleaned CSV files in `data/processed/`

**Steps:**
1. Load raw data
2. Drop movies with no ratings (cold-start items)
3. Merge orphaned ratings before deduplication
4. Deduplicate movies with the same tmdbId
5. Drop movies with no genre information (and their ratings)
6. Parse JSON columns (genres, keywords) into flat lists
7. Compute per-movie rating statistics
8. Merge movies + ratings + credits into unified tables
9. Subsample ratings via random user sampling
10. Validate and save processed outputs

## 1. Load Libraries and Raw Data

In [ ]:
import pandas as pd
import json
import os

In [ ]:
movies_raw = pd.read_csv('../data/raw/tmdb_movie_dataset.csv')
ratings_raw = pd.read_csv('../data/raw/tmdb_movie_ratings.csv')
credits_raw = pd.read_csv('../data/raw/tmdb_movie_credits.csv')

print(f'Loaded: movies={movies_raw.shape}, ratings={ratings_raw.shape}, credits={credits_raw.shape}')

## 2. Drop Movies With No Ratings (Cold-Start Items)

**What we do:** Remove 7 movies whose `ratingId` does not appear in the ratings file.

**Why (literature):** Items with zero interactions are called *cold-start items*. Collaborative filtering requires user-item interaction history to compute similarity or learn latent factors.

> Schein, A. I., Popescul, A., Ungar, L. H., & Pennock, D. M. (2002). Methods and metrics for cold-start recommendations. *Proceedings of the 25th Annual International ACM SIGIR Conference on Research and Development in Information Retrieval*, 253–260.
>
> Lika, B., Kolomvatsos, K., & Hadjiefthymiades, S. (2014). Facing the cold start problem in recommender systems. *Expert Systems with Applications*, 41(4), 2065–2073.

In [ ]:
valid_rating_ids = set(ratings_raw['ratingId'])
movies = movies_raw[movies_raw['ratingId'].isin(valid_rating_ids)].copy()

dropped = len(movies_raw) - len(movies)
print(f'Dropped {dropped} cold-start movies')
print(f'Movies remaining: {len(movies)}')

## 3. Merge Orphaned Ratings Before Deduplication

**What we do:** 4 movies have duplicate `tmdbId` entries with different `ratingId`. We reassign orphaned ratings to the winning `ratingId` so no data is lost.

In [ ]:
dup_mask = movies.duplicated(subset='tmdbId', keep=False)
dup_movies = movies[dup_mask][['tmdbId', 'title', 'ratingId']].sort_values('tmdbId')

print('=== Duplicate tmdbId entries ===')
print(dup_movies.to_string())

rating_counts = ratings_raw.groupby('ratingId').size().reset_index(name='num_ratings')
dup_with_counts = dup_movies.merge(rating_counts, on='ratingId', how='left')

reassign_map = {}
for tmdb_id, group in dup_with_counts.groupby('tmdbId'):
    gs = group.sort_values('num_ratings', ascending=False)
    for losing in gs.iloc[1:]['ratingId'].tolist():
        reassign_map[losing] = gs.iloc[0]['ratingId']

ratings_clean = ratings_raw.copy()
ratings_clean['ratingId'] = ratings_clean['ratingId'].replace(reassign_map)
print(f'Reassigned {len(reassign_map)} orphaned ratingId mappings')

## 4. Deduplicate Movies by tmdbId

**Why (literature):** Duplicate records violate data integrity and can bias model training.

> Ilyas, I. F., & Chu, X. (2015). Trends in cleaning relational data: Consistency and deduplication. *Foundations and Trends in Databases*, 5(4), 283–399.

In [ ]:
rc = ratings_clean.groupby('ratingId').size().reset_index(name='num_ratings')
movies_with_counts = movies.merge(rc, on='ratingId', how='left')
movies = movies_with_counts.sort_values('num_ratings', ascending=False).drop_duplicates(subset='tmdbId', keep='first').drop(columns=['num_ratings']).reset_index(drop=True)

print(f'After dedup: {len(movies)} movies')

## 5. Drop Movies With No Genre Information

**What we do:** Remove movies whose `genres` field is empty. Also drop their corresponding ratings.

In [ ]:
def parse_json_names(json_str):
    try:
        items = json.loads(json_str)
        return [item['name'] for item in items]
    except (json.JSONDecodeError, TypeError, KeyError):
        return []

movies['genre_list'] = movies['genres'].apply(parse_json_names)

empty_mask = movies['genre_list'].apply(len) == 0
if empty_mask.sum() > 0:
    dropped_rids = set(movies[empty_mask]['ratingId'])
    n_before = len(ratings_clean)
    ratings_clean = ratings_clean[~ratings_clean['ratingId'].isin(dropped_rids)]
    movies = movies[~empty_mask].reset_index(drop=True)
    print(f'Dropped {empty_mask.sum()} empty-genre movies + {n_before - len(ratings_clean)} ratings')

print(f'Movies: {len(movies)}, Ratings: {len(ratings_clean):,}')

## 6. Parse JSON Columns

**Why (literature):** Content-based and hybrid recommender systems rely on item metadata to model item characteristics.

> Lops, P., De Gemmis, M., & Semeraro, G. (2010). Content-based recommender systems: State of the art and trends. In P. B. (Ed.), *Recommender Systems Handbook* (pp. 73–105). Springer.

In [ ]:
movies['keyword_list'] = movies['keywords'].apply(parse_json_names)
movies['genres_str'] = movies['genre_list'].apply(lambda x: '|'.join(x))
print(movies[['title', 'genres_str']].head(3).to_string())

## 7. Compute Per-Movie Rating Statistics

> Adomavicius, G., & Zhang, J. (2012). Impact of data characteristics on recommender systems performance. *ACM Transactions on Management Information Systems*, 3(1), 1–23.

In [ ]:
movie_stats = ratings_clean.groupby('ratingId').agg(
    avg_rating=('rating', 'mean'),
    num_ratings=('rating', 'count'),
    rating_std=('rating', 'std')
).reset_index()
movie_stats['avg_rating'] = movie_stats['avg_rating'].round(2)
movie_stats['rating_std'] = movie_stats['rating_std'].round(2)
movies = movies.merge(movie_stats, on='ratingId', how='left')
print(movies[['title', 'avg_rating', 'num_ratings']].head(3).to_string())

## 8. Merge All Tables Into a Unified Dataset

In [ ]:
movies_full = movies.merge(credits_raw[['tmdbId', 'cast', 'crew']], on='tmdbId', how='left')

ratings_full = ratings_clean.merge(
    movies[['ratingId', 'tmdbId', 'title', 'genres_str', 'genre_list']],
    on='ratingId', how='left'
)

print(f'movies_full: {movies_full.shape}, ratings_full: {ratings_full.shape}')
print(f'Missing titles: {ratings_full["title"].isnull().sum()}')

## 9. Subsample Ratings via Random User Sampling

**What we do:** Randomly select 5,000 users and keep all their ratings. This reduces the dataset from ~17M to ~1.5M ratings.

**Why (literature):**
1. **Practical constraint:** The full 17.2M rating dataset is slow to load, difficult to iterate on, and exceeds GitHub's file size limit. Subsampling is standard practice for academic and development work.
2. **Benchmark convention:** The most widely used recommender system benchmarks (MovieLens 100K, MovieLens 1M, MovieLens 25M) are all subsampled versions of larger datasets, chosen to balance coverage with computational feasibility.
3. **Random sampling preserves distribution:** Unlike threshold filtering, random user sampling maintains the natural distribution of rating behaviour — active and casual users are both represented proportionally.

> Harper, F. M., & Konstan, J. A. (2015). The MovieLens datasets: History and context. *ACM Transactions on Interactive Intelligent Systems*, 5(4), 1–19.

The authors of the MovieLens benchmarks explicitly describe subsampling as a methodology for creating practical, reproducible evaluation datasets. Our approach follows the same principle.

In [ ]:
# Number of users to sample — adjust if needed
N_USERS = 2000

print(f'=== Before subsampling ===')
print(f'Ratings: {len(ratings_full):,}, Users: {ratings_full["userId"].nunique():,}, Movies: {ratings_full["ratingId"].nunique():,}')
print()

# Randomly sample N_USERS users (reproducible via random_state)
sampled_user_ids = ratings_full['userId'].sample(n=N_USERS, random_state=42).unique()
ratings_full = ratings_full[ratings_full['userId'].isin(sampled_user_ids)].copy()

print(f'=== After subsampling ({N_USERS} random users) ===')
print(f'Ratings: {len(ratings_full):,}, Users: {ratings_full["userId"].nunique():,}, Movies: {ratings_full["ratingId"].nunique():,}')
print(f'Avg ratings per user: {len(ratings_full) / N_USERS:.0f}')
print()

# Update movies_full to only include movies that remain after subsampling
remaining_rids = set(ratings_full['ratingId'])
movies_full = movies_full[movies_full['ratingId'].isin(remaining_rids)].copy()

# Recompute rating stats for subsampled data
ms_sub = ratings_full.groupby('ratingId').agg(
    avg_rating=('rating', 'mean'), num_ratings=('rating', 'count'), rating_std=('rating', 'std')
).reset_index()
ms_sub['avg_rating'] = ms_sub['avg_rating'].round(2)
ms_sub['rating_std'] = ms_sub['rating_std'].round(2)
movies_full = movies_full.drop(columns=['avg_rating', 'num_ratings', 'rating_std'], errors='ignore').merge(ms_sub, on='ratingId', how='left')

print(f'movies_full updated: {movies_full.shape}')

## 10. Sparsity

> Koren, Y., Bell, R., & Volinsky, C. (2009). Matrix factorization techniques for recommender systems. *Computer*, 42(8), 30–37.

In [ ]:
nu = ratings_full['userId'].nunique()
nm = ratings_full['ratingId'].nunique()
sp = 1 - len(ratings_full) / (nu * nm)
print(f'Matrix: {nu:,} users x {nm:,} movies')
print(f'Ratings: {len(ratings_full):,}, Sparsity: {sp:.2%}')

## 11. Final Validation

In [ ]:
print(f'movies_full: {movies_full.shape}')
print(f'  Unique tmdbId: {movies_full["tmdbId"].nunique()}')
print(f'  Missing avg_rating: {movies_full["avg_rating"].isnull().sum()}')
print(f'  Empty genres: {(movies_full["genre_list"].apply(len) == 0).sum()}')
print()
print(f'ratings_full: {ratings_full.shape}')
print(f'  Unique users: {ratings_full["userId"].nunique():,}')
print(f'  Unique movies: {ratings_full["ratingId"].nunique():,}')
print(f'  Missing titles: {ratings_full["title"].isnull().sum()}')
print(f'  Missing genres: {ratings_full["genres_str"].isnull().sum()}')
print(f'  Duplicates: {ratings_full.duplicated(subset=["userId","ratingId","rating"]).sum()}')

## 12. Save Processed Data

In [ ]:
os.makedirs('../data/processed', exist_ok=True)

movies_full.to_csv('../data/processed/movies_clean.csv', index=False)
ratings_full.to_csv('../data/processed/ratings_clean.csv', index=False)
movies_full[['ratingId','tmdbId','title','genres_str','genre_list','avg_rating','num_ratings']].to_csv('../data/processed/movie_lookup.csv', index=False)

mood_to_genres = {
    'Happy': ['Comedy', 'Animation', 'Family', 'Music'],
    'Sad': ['Drama', 'Romance', 'Comedy'],
    'Stressed': ['Comedy', 'Animation', 'Family', 'Documentary'],
    'Excited': ['Action', 'Adventure', 'Thriller', 'Science Fiction'],
    'Romantic': ['Romance', 'Drama', 'Comedy'],
    'Bored': ['Adventure', 'Action', 'Science Fiction', 'Mystery', 'Horror'],
}
mood_rows = [{'mood': m, 'genres': '|'.join(g)} for m, g in mood_to_genres.items()]
pd.DataFrame(mood_rows).to_csv('../data/processed/mood_genre_mapping.csv', index=False)

for f in ['movies_clean.csv','ratings_clean.csv','movie_lookup.csv','mood_genre_mapping.csv']:
    sz = os.path.getsize(f'../data/processed/{f}') / (1024*1024)
    print(f'Saved {f}: {sz:.1f} MB')

## 13. Preprocessing Summary

| Step | Action | Justification | Reference |
|---|---|---|---|
| Drop unrated movies | Removed 7 cold-start movies | No collaborative signal | Schein et al. (2002); Lika et al. (2014) |
| Reassign orphaned ratings | Merged 315 ratings from duplicates | Prevents data loss | — |
| Deduplicate tmdbId | Kept entry with most ratings | Duplicate records distort similarity | Ilyas & Chu (2015) |
| Drop empty-genre movies | Removed 11 movies + 64 ratings | Cannot participate in mood mapping | Lops et al. (2010) |
| Parse genres/keywords | Converted JSON to lists | Feature extraction | Lops et al. (2010) |
| Compute rating stats | avg_rating, num_ratings, rating_std | Rating distributions affect performance | Adomavicius & Zhang (2012) |
| Subsample ratings | Random 5,000 users | Practical size; benchmark convention | Harper & Konstan (2015) |
| Mood-genre mapping | 6 moods mapped to genres | Mood management theory | Zillmann (1988); Winoto & Tang (2010) |

### Output Files

| File | Rows | Size | Use Case |
|---|---|---|---|
| `movies_clean.csv` | ~4,500 | ~43 MB | Content-based filtering |
| `ratings_clean.csv` | ~1.5M | ~70 MB | Collaborative filtering |
| `movie_lookup.csv` | ~4,500 | <1 MB | Quick reference |
| `mood_genre_mapping.csv` | 6 | <1 KB | Mood-to-genre mapping |

### References

1. Schein, A. I., Popescul, A., Ungar, L. H., & Pennock, D. M. (2002). Methods and metrics for cold-start recommendations. *SIGIR*, 253–260.
2. Lika, B., Kolomvatsos, K., & Hadjiefthymiades, S. (2014). Facing the cold start problem in recommender systems. *Expert Systems with Applications*, 41(4), 2065–2073.
3. Ilyas, I. F., & Chu, X. (2015). Trends in cleaning relational data: Consistency and deduplication. *Foundations and Trends in Databases*, 5(4), 283–399.
4. Lops, P., De Gemmis, M., & Semeraro, G. (2010). Content-based recommender systems. In *Recommender Systems Handbook* (pp. 73–105). Springer.
5. Adomavicius, G., & Zhang, J. (2012). Impact of data characteristics on recommender systems performance. *ACM TMIS*, 3(1), 1–23.
6. Koren, Y., Bell, R., & Volinsky, C. (2009). Matrix factorization techniques for recommender systems. *Computer*, 42(8), 30–37.
7. Harper, F. M., & Konstan, J. A. (2015). The MovieLens datasets: History and context. *ACM TiiS*, 5(4), 1–19.
8. Zillmann, D. (1988). Mood management through communication choices. *American Behavioral Scientist*, 31(3), 327–340.
9. Winoto, P., & Tang, T. Y. (2010). The role of user mood in movie recommendations. *Expert Systems with Applications*, 37(8), 6086–6092.